# Viterbi Algorithm

The Viterbi algorithm finds the most likely sequence of hidden states for a sequence of observations in a Hidden Markov Model (HMM). This notebook implements the algorithm in log space to avoid numerical underflow, then runs a small weather example.

An HMM is specified by initial state probabilities, transition probabilities, and emission probabilities. Each probability array below follows the order in `states` and `observations`.

In [ ]:
import numpy as np

def viterbi(observations, states, start_probability, transition_probability, emission_probability):
    """Return the most likely hidden-state sequence and its log probability."""
    observations = list(observations)
    states = list(states)
    if not observations:
        raise ValueError("observations must contain at least one item")
    if not states:
        raise ValueError("states must contain at least one state")

    n_states = len(states)
    n_steps = len(observations)
    start = np.asarray(start_probability, dtype=float)
    transition = np.asarray(transition_probability, dtype=float)
    emission = np.asarray(emission_probability, dtype=float)
    if start.shape != (n_states,):
        raise ValueError("start_probability must have one value per state")
    if transition.shape != (n_states, n_states):
        raise ValueError("transition_probability must be a square state-by-state matrix")
    if emission.shape[0] != n_states:
        raise ValueError("emission_probability must have one row per state")
    if np.any(start < 0) or np.any(transition < 0) or np.any(emission < 0):
        raise ValueError("probabilities cannot be negative")

    # Map each observation to its column in the emission matrix.
    observation_index = {observation: i for i, observation in enumerate(observation_symbols)}
    try:
        obs_indices = [observation_index[item] for item in observations]
    except (KeyError, TypeError) as exc:
        raise ValueError(f"unknown observation: {exc.args[0]}") from exc
    if emission.shape[1] != len(observation_symbols):
        raise ValueError("emission matrix columns must match observation_symbols")

    # log(0) is -inf, correctly making impossible paths have zero probability.
    with np.errstate(divide="ignore"):
        log_start = np.log(start)
        log_transition = np.log(transition)
        log_emission = np.log(emission)

    scores = np.full((n_steps, n_states), -np.inf)
    backpointers = np.zeros((n_steps, n_states), dtype=int)
    scores[0] = log_start + log_emission[:, obs_indices[0]]

    for t in range(1, n_steps):
        for current in range(n_states):
            candidates = scores[t - 1] + log_transition[:, current]
            backpointers[t, current] = np.argmax(candidates)
            scores[t, current] = (
                candidates[backpointers[t, current]]
                + log_emission[current, obs_indices[t]]
            )

    last_state = int(np.argmax(scores[-1]))
    best_log_probability = float(scores[-1, last_state])
    path_indices = [last_state]
    for t in range(n_steps - 1, 0, -1):
        path_indices.append(int(backpointers[t, path_indices[-1]]))
    path_indices.reverse()
    return [states[i] for i in path_indices], best_log_probability

## Example: inferring weather from activities

The hidden states are `Rainy` and `Sunny`; the observations are `walk`, `shop`, and `clean`. The returned log probability is the log of the joint probability of the best hidden path and the observations.

In [ ]:
states = ["Rainy", "Sunny"]
observation_symbols = ["walk", "shop", "clean"]
observations = ["walk", "shop", "clean"]

start_probability = [0.6, 0.4]
transition_probability = [
    [0.7, 0.3],  # From Rainy to [Rainy, Sunny]
    [0.4, 0.6],  # From Sunny to [Rainy, Sunny]
]
emission_probability = [
    [0.1, 0.4, 0.5],  # Rainy: P(walk), P(shop), P(clean)
    [0.6, 0.3, 0.1],  # Sunny: P(walk), P(shop), P(clean)
]

best_path, log_probability = viterbi(
    observations,
    states,
    start_probability,
    transition_probability,
    emission_probability,
)
print("Most likely weather sequence:", best_path)
print("Log probability of this path and observations:", log_probability)

## How it works

1. Initialize each state's score using its start and first-observation probabilities.
2. At each subsequent observation, choose for each current state the previous state that maximizes the accumulated log probability. Save that choice in a backpointer.
3. Select the highest-scoring final state and follow the backpointers backward to recover the best sequence.

The dynamic-programming computation takes O(T × N²) time and O(T × N) memory, where T is the number of observations and N is the number of states.